# Chapter 1 — Figure Replication
## Univariate Time Series: From Stationarity to Cointegration

**Macroeconometrics** | Alessia Paccagnini | Companion Notebook

---

## What this notebook does

This notebook replicates **all figures** from Chapter 1, grouped into four thematic blocks that follow the chapter's logical progression:

| Block | Topic | Figures |
|---|---|---|
| **A** | ARMA processes — simulation, ACF, PACF | 3 figures |
| **B** | Nonstationary processes — trends, random walks, variance | 7 figures |
| **C** | Spurious regression — single examples, Monte Carlo | 3 figures |
| **D** | Spurious regression — Granger-Newbold rule, t-stat distributions | 2 figures |

Every code cell is annotated with two layers:
- **Narrative text blocks** (markdown cells like this one) explain *why* each step is done and connect it to the theory in the textbook.
- **Inline comments** inside the code explain *what* each line does.

**How to use this notebook:** Run cells in order from top to bottom. All figures are saved to disk automatically (`.png` and `.pdf`). You can modify the parameter cells — clearly marked with 🎛️ — to experiment with how the figures change.

---

## Dependencies

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.api as sm
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.stattools import acf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.stattools import durbin_watson
from scipy import stats
import warnings, os
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
# Setting the seed means every run produces identical figures.
# Change the seed to see how results vary across random draws.
np.random.seed(42)

# ── Output directory ──────────────────────────────────────────────────────────
# Figures are saved here. Change this path if you want them elsewhere.
OUT_DIR = '.'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Plot style ────────────────────────────────────────────────────────────────
# These settings produce clean, publication-quality figures.
plt.rcParams.update({
    'figure.dpi': 130,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'legend.fontsize': 10,
})

print('Setup complete. All packages loaded.')

---
# Block A — ARMA Processes
### *Textbook reference: Sections 1.3–1.5*

## Background

ARMA models are the foundational building blocks of univariate time series analysis. The general ARMA(p,q) model is:

$$y_t = c + \phi_1 y_{t-1} + \cdots + \phi_p y_{t-p} + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \cdots + \theta_q \varepsilon_{t-q}$$

where $\varepsilon_t \sim WN(0, \sigma^2)$ is white noise.

The AR part ($\phi$ coefficients) captures **persistence**: how much today's value depends on its own past. The MA part ($\theta$ coefficients) captures **transitory shocks**: how long an individual disturbance echoes through the series.

**Why simulate?** Simulating ARMA processes with known parameters and then computing their ACF/PACF is the best way to build intuition for the Box-Jenkins identification patterns you will use when working with real data.

## 🎛️ Parameters — modify these to experiment

In [ ]:
# 🎛️ ARMA PARAMETERS — change these and re-run all Block A cells
T_ARMA  = 500    # Sample size (more observations → smoother sample ACF/PACF)
SIGMA   = 1.0    # White noise standard deviation
PHI     = 0.7    # AR(1) coefficient  — try 0.3 (weak) or 0.95 (near unit root)
THETA   = 0.5    # MA(1) coefficient  — try -0.8 or 0.9

# Stationarity check: |phi| < 1 is required for AR/ARMA stability
assert abs(PHI) < 1, f"AR coefficient must satisfy |phi| < 1 for stationarity. Got PHI={PHI}"
print(f'ARMA parameters: T={T_ARMA}, sigma={SIGMA}, phi={PHI}, theta={THETA}')

## Simulation functions

We simulate each process **manually** using explicit loops. This is pedagogically important: you can see exactly how each data point is generated. The `statsmodels` library has built-in ARMA simulators (faster, used for verification), but the manual code makes the data-generating process transparent.

**Initialisation note for AR(1):** The unconditional variance of a stationary AR(1) is $\sigma^2/(1-\phi^2)$. We initialise $y_0$ by drawing from this distribution rather than setting $y_0 = 0$. This means the series is already in its steady-state distribution at $t=0$, avoiding a burn-in period.

In [ ]:
# ── ARMA simulation functions ─────────────────────────────────────────────────

def generate_white_noise(T, sigma=1.0):
    """White noise: each draw is i.i.d. N(0, sigma^2). No memory whatsoever."""
    return np.random.normal(0, sigma, T)


def generate_ar1(T, phi, sigma=1.0):
    """
    AR(1): y_t = phi * y_{t-1} + eps_t
    Requires |phi| < 1 for stationarity (Section 1.3.1).
    """
    eps = np.random.normal(0, sigma, T)       # draw all shocks at once (faster)
    y   = np.zeros(T)
    # Initialise from the unconditional distribution — avoids burn-in bias
    y[0] = np.random.normal(0, sigma / np.sqrt(1 - phi**2))
    for t in range(1, T):
        y[t] = phi * y[t-1] + eps[t]          # recursion: each obs depends on previous
    return y, eps


def generate_ma1(T, theta, sigma=1.0):
    """
    MA(1): y_t = eps_t + theta * eps_{t-1}
    Always stationary for any finite theta (Section 1.4.1).
    We need T+1 shocks because y_1 depends on eps_0.
    """
    eps = np.random.normal(0, sigma, T + 1)   # one extra shock for the initial lag
    y   = np.array([eps[t+1] + theta * eps[t] for t in range(T)])
    return y, eps[1:]                         # return T shocks aligned with T obs


def generate_arma11(T, phi, theta, sigma=1.0):
    """
    ARMA(1,1): y_t = phi * y_{t-1} + eps_t + theta * eps_{t-1}
    Combines AR persistence with MA transitory shock.
    Stationary iff |phi| < 1.
    """
    eps = np.random.normal(0, sigma, T + 1)
    y   = np.zeros(T)
    y[0] = eps[1] + theta * eps[0]            # initialise: no lagged y available
    for t in range(1, T):
        y[t] = phi * y[t-1] + eps[t+1] + theta * eps[t]   # AR + MA components
    return y, eps[1:]


# ── Generate all four processes ───────────────────────────────────────────────
wn    = generate_white_noise(T_ARMA, SIGMA)
ar1, eps_ar1     = generate_ar1(T_ARMA, PHI, SIGMA)
ma1, eps_ma1     = generate_ma1(T_ARMA, THETA, SIGMA)
arma11, eps_arma = generate_arma11(T_ARMA, PHI, THETA, SIGMA)

# Quick sanity check: theoretical vs sample variance
print('Theoretical vs sample variance (should be close with T=500):')
print(f'  White noise:  theory={SIGMA**2:.3f},  sample={np.var(wn):.3f}')
print(f'  AR(1):        theory={SIGMA**2/(1-PHI**2):.3f},  sample={np.var(ar1):.3f}')
print(f'  MA(1):        theory={SIGMA**2*(1+THETA**2):.3f},  sample={np.var(ma1):.3f}')
arma_var = SIGMA**2 * (1 + THETA**2 + 2*PHI*THETA) / (1 - PHI**2)
print(f'  ARMA(1,1):    theory={arma_var:.3f},  sample={np.var(arma11):.3f}')

**What you should see:** Sample variances track theoretical values closely at $T=500$. Try reducing `T_ARMA` to 50 and re-running — the gap grows substantially, illustrating why small-sample estimation is harder.

## Figure A-1 — Time series plots

The four panels show each process plotted over time. Key visual features to identify:

- **White noise**: centred at zero, constant variance, *no* visible pattern — each observation looks independent.
- **AR(1)**: persistent runs above or below zero because $\phi=0.7$ means 70% of last period's level carries forward. With $\phi=0.95$, the series looks almost like a random walk.
- **MA(1)**: less persistent than AR(1). Each shock lasts only one period into the future because $\theta$ only multiplies $\varepsilon_{t-1}$.
- **ARMA(1,1)**: combines both effects — persistence from the AR part, shock propagation from the MA part.

In [ ]:
# ── Figure A-1: ARMA time series ──────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(13, 11), sharex=True)

# Series, colors, labels, and LaTeX titles
series_spec = [
    (wn,    'steelblue',  r'$\varepsilon_t \sim N(0,1)$',
     r'White Noise: $\varepsilon_t \sim N(0,\sigma^2)$'),
    (ar1,   'darkgreen',  r'$y_t = 0.7\,y_{t-1} + \varepsilon_t$',
     rf'AR(1): $y_t = {PHI}\,y_{{t-1}} + \varepsilon_t$'),
    (ma1,   'darkorange', r'$y_t = \varepsilon_t + 0.5\,\varepsilon_{t-1}$',
     rf'MA(1): $y_t = \varepsilon_t + {THETA}\,\varepsilon_{{t-1}}$'),
    (arma11,'purple',     r'$y_t = 0.7\,y_{t-1} + \varepsilon_t + 0.5\,\varepsilon_{t-1}$',
     rf'ARMA(1,1): $y_t = {PHI}\,y_{{t-1}} + \varepsilon_t + {THETA}\,\varepsilon_{{t-1}}$'),
]

for ax, (series, color, _, title) in zip(axes, series_spec):
    ax.plot(series, color=color, linewidth=0.85, alpha=0.9)
    ax.axhline(0, color='red', linestyle='--', linewidth=1.0, alpha=0.6)
    ax.set_title(title)
    ax.set_ylabel(r'$y_t$')

axes[-1].set_xlabel('Time $t$')
fig.suptitle('Figure A-1: ARMA Process Simulations\n'
             f'($T={T_ARMA}$, $\\phi={PHI}$, $\\theta={THETA}$, seed=42)',
             fontsize=13, fontweight='bold')
fig.tight_layout()

path = os.path.join(OUT_DIR, 'figA1_arma_time_series.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png', '.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

## Figure A-2 — ACF comparison

The **Autocorrelation Function (ACF)** at lag $k$ measures the correlation between $y_t$ and $y_{t-k}$. It is the primary tool for identifying the MA order $q$ in Box-Jenkins identification (Section 1.5.2).

**The identification patterns (Table 1.X in the textbook):**

| Process | ACF pattern |
|---|---|
| White noise | No significant spikes beyond lag 0 |
| AR(1) | Geometric decay: $\rho_k = \phi^k$ — never cuts off sharply |
| MA(1) | Single spike at lag 1, then cuts off to zero |
| ARMA(1,1) | Geometric decay *after* lag 1 (a mix of both patterns) |

The blue shaded region is the 95% confidence band under the null of white noise: $\pm 1.96/\sqrt{T}$. Spikes outside this band are statistically significant at the 5% level. Notice: the band gets narrower as $T$ increases — larger samples give sharper identification.

In [ ]:
# ── Figure A-2: ACF comparison ────────────────────────────────────────────────
N_LAGS = 20     # number of lags to display

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

acf_spec = [
    (wn,    'White Noise'),
    (ar1,   rf'AR(1), $\phi={PHI}$'),
    (ma1,   rf'MA(1), $\theta={THETA}$'),
    (arma11,rf'ARMA(1,1), $\phi={PHI}$, $\theta={THETA}$'),
]

for ax, (series, title) in zip(axes.flatten(), acf_spec):
    # plot_acf draws the ACF bars and 95% confidence bands automatically
    plot_acf(series, ax=ax, lags=N_LAGS,
             title=f'ACF: {title}',
             zero=False,          # skip lag-0 (always = 1, not informative)
             alpha=0.05)          # 95% confidence bands
    ax.set_xlabel('Lag $k$')
    ax.set_ylabel(r'$\hat{\rho}_k$')
    ax.axhline(0, color='black', linewidth=0.7)

fig.suptitle('Figure A-2: Sample ACF — ARMA Identification Patterns\n'
             '(blue band = 95% CI under white noise null)',
             fontsize=13, fontweight='bold')
fig.tight_layout()

path = os.path.join(OUT_DIR, 'figA2_arma_acf.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png', '.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

## Figure A-3 — PACF comparison

The **Partial Autocorrelation Function (PACF)** at lag $k$ measures the correlation between $y_t$ and $y_{t-k}$ *after removing the linear effect of all intermediate lags* $y_{t-1}, \ldots, y_{t-k+1}$. It is the primary tool for identifying the AR order $p$.

**The identification patterns:**

| Process | PACF pattern |
|---|---|
| White noise | No significant spikes |
| AR(1) | Single spike at lag 1, then cuts off |
| MA(1) | Geometric decay — never cuts off sharply |
| ARMA(1,1) | Geometric decay *after* lag 1 |

**ACF + PACF together:** Use ACF to find $q$ (where it cuts off), PACF to find $p$ (where it cuts off). When both decay geometrically, you have an ARMA process — neither order is obvious from the correlograms alone, and information criteria (AIC/BIC) become necessary.

We use the Yule-Walker (`ywm`) method for PACF estimation, which is numerically stable at all lags.

In [ ]:
# ── Figure A-3: PACF comparison ───────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

for ax, (series, title) in zip(axes.flatten(), acf_spec):
    plot_pacf(series, ax=ax, lags=N_LAGS,
              title=f'PACF: {title}',
              zero=False,
              alpha=0.05,
              method='ywm')    # Yule-Walker modified — preferred for short lags
    ax.set_xlabel('Lag $k$')
    ax.set_ylabel(r'$\hat{\alpha}_k$')
    ax.axhline(0, color='black', linewidth=0.7)

fig.suptitle('Figure A-3: Sample PACF — ARMA Identification Patterns\n'
             '(blue band = 95% CI under white noise null)',
             fontsize=13, fontweight='bold')
fig.tight_layout()

path = os.path.join(OUT_DIR, 'figA3_arma_pacf.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png', '.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

# ── Identification summary table ───────────────────────────────────────────────
print('\nBox-Jenkins Identification Guide:')
print(f'{"Process":12s} {"ACF":32s} {"PACF"}')
print('-'*72)
rows = [
    ('White noise', 'No significant spikes',        'No significant spikes'),
    ('AR(p)',       'Geometric decay',               'Cuts off after lag p'),
    ('MA(q)',       'Cuts off after lag q',           'Geometric decay'),
    ('ARMA(p,q)',   'Decays (after lag q)',           'Decays (after lag p)'),
]
for proc, acf_pat, pacf_pat in rows:
    print(f'{proc:12s} {acf_pat:32s} {pacf_pat}')

---
# Block B — Nonstationary Processes
### *Textbook reference: Sections 1.6–1.8*

## Background

Not all time series are stationary. Macroeconomic variables like GDP, price levels, and exchange rates typically contain **unit roots** — their unconditional variance grows over time. Chapter 1 distinguishes three fundamentally different ways a series can be nonstationary:

| Type | Equation | How to stationarise |
|---|---|---|
| **Deterministic trend** | $y_t = \alpha + \beta t + u_t$, $u_t \sim$ AR(1) | Detrend (subtract $\hat{\alpha} + \hat{\beta}t$) |
| **Random walk** | $y_t = y_{t-1} + \varepsilon_t$ | First-difference: $\Delta y_t = \varepsilon_t$ |
| **RW with drift** | $y_t = \delta + y_{t-1} + \varepsilon_t$ | First-difference: $\Delta y_t = \delta + \varepsilon_t$ |

**Why the distinction matters:** If you difference a trend-stationary series, you *over-difference* it and introduce negative MA structure. If you detrend a unit-root series, the residual still has a unit root. Getting the transformation wrong affects all downstream analysis.

## 🎛️ Parameters

In [ ]:
# 🎛️ NONSTATIONARY PROCESS PARAMETERS
T_NS   = 250    # sample size
ALPHA  = 2.0    # deterministic trend: intercept
BETA   = 0.1    # deterministic trend: slope — try 0.0 (no trend) or 0.3 (steep)
PHI_U  = 0.7    # AR(1) coef for stationary component around the trend
DELTA  = 0.15   # drift in the random walk with drift
N_SIMS = 100    # number of MC replications for the variance comparison figure

print(f'Parameters: T={T_NS}, alpha={ALPHA}, beta={BETA}, phi_u={PHI_U}, delta={DELTA}')

In [ ]:
# ── Nonstationary simulation functions ───────────────────────────────────────

def generate_deterministic_trend(T, alpha, beta, phi, sigma=1.0):
    """
    Trend-stationary: y_t = alpha + beta*t + u_t, u_t = phi*u_{t-1} + eps_t
    This is NOT a unit root — detrending yields a stationary series.
    """
    eps = np.random.normal(0, sigma, T)
    u   = np.zeros(T)
    u[0] = np.random.normal(0, sigma / np.sqrt(1 - phi**2))  # draw from unconditional dist
    for t in range(1, T):
        u[t] = phi * u[t-1] + eps[t]
    t_idx = np.arange(T)
    y = alpha + beta * t_idx + u       # add deterministic trend on top of AR(1)
    return y, u, t_idx


def generate_random_walk(T, sigma=1.0, y0=0):
    """
    Pure random walk: y_t = y_{t-1} + eps_t
    Var(y_t) = t*sigma^2 — grows without bound.
    E[y_t] = y_0 — no tendency to revert to any level.
    """
    eps = np.random.normal(0, sigma, T)
    y   = np.zeros(T)
    y[0] = y0 + eps[0]
    for t in range(1, T):
        y[t] = y[t-1] + eps[t]         # today = yesterday + new shock
    return y, eps


def generate_random_walk_drift(T, delta, sigma=1.0, y0=0):
    """
    RW with drift: y_t = delta + y_{t-1} + eps_t
    E[y_t] = y_0 + delta*t — grows at rate delta.
    Var(y_t) = t*sigma^2 — still grows without bound.
    Looks like a trend but cannot be stationarised by detrending alone.
    """
    eps = np.random.normal(0, sigma, T)
    y   = np.zeros(T)
    y[0] = y0 + delta + eps[0]
    for t in range(1, T):
        y[t] = delta + y[t-1] + eps[t] # constant drift added each period
    return y, eps


# ── Generate all three nonstationary series ───────────────────────────────────
np.random.seed(42)   # reset seed for reproducibility of nonstationary block
y_det, u_det, t_idx = generate_deterministic_trend(T_NS, ALPHA, BETA, PHI_U)
y_rw,  eps_rw       = generate_random_walk(T_NS)
y_rwd, eps_rwd      = generate_random_walk_drift(T_NS, DELTA)
wn_ns               = np.random.normal(0, 1, T_NS)   # white noise for ACF reference

print('Generated: deterministic trend, random walk, random walk with drift.')

## Figure B-1 — Three nonstationary processes side by side

This panel figure is the key visual for understanding the qualitative difference between the three types of nonstationarity. Look for:

- **Top panel**: series oscillates around the trend line (red dashes) — the gap is bounded, confirming the stationary AR(1) component.
- **Middle panel**: no tendency to return to zero — drifts freely in either direction, with increasing variance over time (the fill shows how wide the excursions become).
- **Bottom panel**: looks like the deterministic trend but isn't — the upward drift is stochastic, not deterministic. Removing the trend line $\delta t$ does *not* make this series stationary.

In [ ]:
# ── Figure B-1: Three nonstationary processes ────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

# Panel 1: Deterministic trend
axes[0].plot(y_det, color='steelblue', linewidth=0.9, label='Series $y_t$')
axes[0].plot(t_idx, ALPHA + BETA * t_idx, color='red', linestyle='--',
             linewidth=2.0, label=fr'Trend: ${ALPHA} + {BETA}t$')
axes[0].set_title(fr'Deterministic Trend: $y_t = {ALPHA} + {BETA}t + u_t$,  $u_t = {PHI_U}u_{{t-1}} + \varepsilon_t$')
axes[0].set_ylabel(r'$y_t$')
axes[0].legend(loc='upper left')

# Panel 2: Random walk (fill shows growing range of variation)
axes[1].plot(y_rw, color='darkgreen', linewidth=0.9)
axes[1].fill_between(range(T_NS), y_rw, alpha=0.25, color='darkgreen')
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5,
                alpha=0.7, label='Starting level $y_0 = 0$')
axes[1].set_title(r'Random Walk: $y_t = y_{t-1} + \varepsilon_t$  (variance $= t\sigma^2$ grows linearly)')
axes[1].set_ylabel(r'$y_t$')
axes[1].legend(loc='best')

# Panel 3: RW with drift
axes[2].plot(y_rwd, color='purple', linewidth=0.9, label='Series $y_t$')
axes[2].plot(t_idx, DELTA * t_idx, color='red', linestyle='--',
             linewidth=2.0, label=fr'Drift component: ${DELTA}t$')
axes[2].set_title(fr'Random Walk with Drift: $y_t = {DELTA} + y_{{t-1}} + \varepsilon_t$')
axes[2].set_ylabel(r'$y_t$')
axes[2].set_xlabel('Time $t$')
axes[2].legend(loc='upper left')

fig.suptitle('Figure B-1: Three Types of Nonstationary Processes\n'
             '(red dashes = deterministic component)', fontsize=13, fontweight='bold')
fig.tight_layout()

path = os.path.join(OUT_DIR, 'figB1_nonstationary_comparison.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

## Figure B-2 — ACF: why the correlogram fails for unit roots

The ACF is designed to detect autocorrelation in *stationary* series. For a random walk, the sample ACF at lag $k$ converges to approximately $(T-k)/T \to 1$, not to zero. **All lags look nearly equally correlated** — the correlogram gives no useful information about the order of the process.

This is why formal unit root tests (ADF, KPSS) are necessary. You cannot identify a near-unit-root AR(1) with $\phi=0.99$ from a true unit root $\phi=1.00$ just by looking at the correlogram.

In [ ]:
# ── Figure B-2: ACF comparison — all four processes ──────────────────────────
N_LAGS_NS = 30

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
ns_acf_spec = [
    (wn_ns, 'White Noise (Stationary)'),
    (y_det, 'Deterministic Trend (before detrending)'),
    (y_rw,  'Random Walk'),
    (y_rwd, 'Random Walk with Drift'),
]

for ax, (series, title) in zip(axes.flatten(), ns_acf_spec):
    plot_acf(series, ax=ax, lags=N_LAGS_NS,
             title=f'ACF: {title}', zero=False, alpha=0.05)
    ax.set_xlabel('Lag $k$')
    ax.set_ylim(-0.3, 1.1)
    ax.axhline(0, color='black', linewidth=0.6)

fig.suptitle('Figure B-2: ACF — Stationary vs. Nonstationary Processes\n'
             'Slow linear decay ≠ high AR order — it signals a unit root',
             fontsize=13, fontweight='bold')
fig.tight_layout()

path = os.path.join(OUT_DIR, 'figB2_nonstationary_acf.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

## Figure B-3 — Levels vs. first differences

The remedy for a random walk is **first-differencing**: $\Delta y_t = y_t - y_{t-1} = \varepsilon_t$, which is white noise. For a RW with drift, $\Delta y_t = \delta + \varepsilon_t$, which is white noise with a non-zero mean.

This figure shows that the ACF in levels has the characteristic slow-decay pattern (unit root), while the ACF of first differences drops immediately to near-zero (white noise). This visual contrast is the diagnostic that tells you differencing was the right transformation.

In [ ]:
# ── Figure B-3: Levels vs first differences ──────────────────────────────────
dy_det = np.diff(y_det)   # first difference of deterministic trend
dy_rw  = np.diff(y_rw)    # first difference of random walk → white noise
dy_rwd = np.diff(y_rwd)   # first difference of RW with drift → white noise + drift

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Top row: ACF in levels
levels_spec = [
    (y_det, 'Deterministic Trend (levels)'),
    (y_rw,  'Random Walk (levels)'),
    (y_rwd, 'RW with Drift (levels)'),
]
diff_spec = [
    (dy_det, 'Det. Trend (first diff.)'),
    (dy_rw,  'Random Walk (first diff.) → WN'),
    (dy_rwd, 'RW + Drift (first diff.) → WN'),
]

for ax, (series, title) in zip(axes[0], levels_spec):
    plot_acf(series, ax=ax, lags=N_LAGS_NS, title=title, zero=False)
    ax.set_ylim(-0.3, 1.1)
    ax.set_xlabel('Lag')

for ax, (series, title) in zip(axes[1], diff_spec):
    plot_acf(series, ax=ax, lags=N_LAGS_NS, title=title, zero=False)
    ax.set_ylim(-0.5, 1.1)
    ax.set_xlabel('Lag')

# Row labels
for ax, label in zip([axes[0,0], axes[1,0]], ['Levels', 'First Differences']):
    ax.set_ylabel(label, fontsize=12, fontweight='bold', labelpad=15)

fig.suptitle('Figure B-3: ACF in Levels vs. First Differences\n'
             'Differencing removes the unit root — ACF drops to near-zero immediately',
             fontsize=13, fontweight='bold')
fig.tight_layout()

path = os.path.join(OUT_DIR, 'figB3_levels_vs_differences.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

## Figure B-4 — Variance growth: random walk vs. stationary AR(1)

This figure provides the cleanest visual proof of the core difference between I(1) and I(0) processes. We simulate $N=100$ paths of each type and compute the **cross-sectional variance** at each time point.

For a random walk: $\text{Var}(y_t) = t\sigma^2$ — grows linearly with time (red dashes).
For a stationary AR(1): $\text{Var}(y_t) = \sigma^2/(1-\phi^2)$ — constant at all $t$.

This is why unit root series cannot be modelled with OLS under standard assumptions: the variance is not bounded, so the $t$-statistics and $F$-statistics do not follow their standard distributions.

In [ ]:
# ── Figure B-4: Variance growth ────────────────────────────────────────────────
T_VAR = 200   # horizon for variance comparison

# Simulate N_SIMS paths of each process
rw_paths  = np.array([generate_random_walk(T_VAR)[0]  for _ in range(N_SIMS)])
ar1_paths = np.array([generate_ar1(T_VAR, PHI_U)[0]   for _ in range(N_SIMS)])

# Cross-sectional variance at each t
rw_var_t  = np.var(rw_paths,  axis=0)     # shape (T_VAR,)
ar1_var_t = np.var(ar1_paths, axis=0)

# Theoretical values
t_axis = np.arange(1, T_VAR + 1)
rw_theory  = t_axis * SIGMA**2                    # linear in t
ar1_theory = SIGMA**2 / (1 - PHI_U**2)            # constant

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Random walk variance
axes[0].plot(t_axis, rw_var_t, color='darkgreen', linewidth=1.5, label='Sample variance')
axes[0].plot(t_axis, rw_theory, color='red', linestyle='--', linewidth=2,
             label=r'Theory: $\text{Var}(y_t) = t\sigma^2$')
axes[0].set_xlabel('Time $t$')
axes[0].set_ylabel('Variance')
axes[0].set_title('Random Walk: Variance Grows Linearly')
axes[0].legend()

# Stationary AR(1) variance
axes[1].plot(t_axis, ar1_var_t, color='steelblue', linewidth=1.5, label='Sample variance')
axes[1].axhline(ar1_theory, color='red', linestyle='--', linewidth=2,
                label=fr'Theory: $\sigma^2/(1-\phi^2) = {ar1_theory:.2f}$')
axes[1].set_xlabel('Time $t$')
axes[1].set_ylabel('Variance')
axes[1].set_title(fr'Stationary AR(1), $\phi={PHI_U}$: Variance Stays Constant')
axes[1].set_ylim(0, ar1_theory * 2.5)
axes[1].legend()

fig.suptitle(f'Figure B-4: Variance Across Time — {N_SIMS} Monte Carlo Paths\n'
             'I(1) variance grows unboundedly; I(0) variance is time-invariant',
             fontsize=13, fontweight='bold')
fig.tight_layout()

path = os.path.join(OUT_DIR, 'figB4_variance_growth.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

## Figure B-5 — Multiple realisations: fan chart

Plotting 30 independent paths of the same DGP gives a powerful visual of what stationarity really means. For the random walk, paths fan out in all directions — no tendency to stay near any fixed level. For the deterministic trend, all paths hover around the same trend line — the variance is bounded.

In [ ]:
# ── Figure B-5: Multiple realisations ────────────────────────────────────────
N_PATHS = 30
np.random.seed(99)   # separate seed so each run shows the same paths

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: random walk paths
for _ in range(N_PATHS):
    path_rw, _ = generate_random_walk(T_NS)
    axes[0].plot(path_rw, alpha=0.35, linewidth=0.8)     # semi-transparent so overlap is visible
axes[0].axhline(0, color='red', linestyle='--', linewidth=2, label='$y_0 = 0$')
axes[0].set_title('Random Walk — Paths Fan Out Over Time')
axes[0].set_xlabel('Time $t$')
axes[0].set_ylabel(r'$y_t$')
axes[0].legend()

# Right: deterministic trend paths
for _ in range(N_PATHS):
    path_dt, _, _ = generate_deterministic_trend(T_NS, ALPHA, BETA, PHI_U)
    axes[1].plot(path_dt, alpha=0.35, linewidth=0.8)
axes[1].plot(t_idx, ALPHA + BETA * t_idx, color='red', linestyle='--',
             linewidth=2.5, label='Deterministic trend')
axes[1].set_title('Deterministic Trend — Paths Stay Near the Trend Line')
axes[1].set_xlabel('Time $t$')
axes[1].set_ylabel(r'$y_t$')
axes[1].legend()

fig.suptitle(f'Figure B-5: Fan Chart — {N_PATHS} Independent Realisations per DGP',
             fontsize=13, fontweight='bold')
fig.tight_layout()

path = os.path.join(OUT_DIR, 'figB5_fan_chart.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

---
# Block C — Spurious Regression: Single Examples
### *Textbook reference: Section 1.7*

## Background

The **spurious regression problem** (Granger & Newbold, 1974; Phillips, 1986) is one of the most important cautionary results in econometrics.

**The setup:** regress $y_t$ on $x_t$ where both are *independent* random walks (no relationship whatsoever). OLS will often find a highly significant coefficient with a high $R^2$. This is not evidence of a real relationship — it is a statistical artifact of the shared non-stationarity.

**Why does this happen?** Both series wander. If they happen to wander in the same direction over the sample, OLS fits a large positive coefficient. The $t$-statistic diverges as $T \to \infty$ rather than converging to zero under the null of no relationship (Phillips, 1986).

**The Granger-Newbold diagnostic:** If $R^2 > DW$ (Durbin-Watson), suspect spurious regression. In genuine spurious regressions, the residuals are highly autocorrelated (DW near 0), reflecting the unit root that OLS failed to remove.

## 🎛️ Parameters

In [ ]:
# 🎛️ SPURIOUS REGRESSION PARAMETERS
T_SPUR       = 300    # sample size for individual spurious regression examples
R2_THRESHOLD = 0.80   # minimum R2 for the 9-panel figure
DW_THRESHOLD = 0.20   # maximum DW for the 9-panel figure
N_SIMS_SPUR  = 1000   # replications for Monte Carlo
N_SIMS_SCATTER = 1000 # replications for Granger-Newbold scatter
N_SIMS_TSTAT = 5000   # replications for t-stat distribution
T_MC         = 200    # sample size used in Monte Carlo figures

print('Spurious regression parameters set.')

In [ ]:
# ── Regression helper ────────────────────────────────────────────────────────
def run_regression(T, spurious=True):
    """
    Run a single OLS regression of y on x.
      spurious=True  → both x and y are independent I(1) random walks
      spurious=False → both x and y are independent I(0) white noise
    Returns: dict with r2, t_stat, dw, beta_hat, fitted values.
    """
    if spurious:
        x = np.cumsum(np.random.randn(T))    # random walk = cumulative sum of WN
        y = np.cumsum(np.random.randn(T))    # independent random walk
    else:
        x = np.random.randn(T)               # white noise
        y = np.random.randn(T)               # independent white noise

    X     = sm.add_constant(x)               # add intercept column
    model = sm.OLS(y, X).fit()

    return {
        'r2':       model.rsquared,
        't_stat':   model.tvalues[1],         # t-stat on the slope
        'dw':       durbin_watson(model.resid),
        'beta_hat': model.params[1],
        'fitted':   model.fittedvalues,
        'x': x, 'y': y
    }

print('Helper function defined.')

## Figure C-1 — Nine spurious regressions

Each panel shows a scatter plot of two *completely independent* random walks, with the OLS fitted line overlaid. The $R^2$, $t$-statistic, and DW statistic are shown in each title.

Notice: all nine regressions look convincing. $R^2 > 0.80$, $|t| > 1.96$ (stars), and DW near zero (highly autocorrelated residuals — the giveaway). A naive researcher might conclude there is a strong, statistically significant relationship in every one of these panels. There isn't.

**The stars next to the $t$-statistic use standard significance thresholds:** `*` (5%), `**` (1%), `***` (0.1%). These thresholds are meaningless here because the $t$-distribution assumption is violated for I(1) series.

In [ ]:
# ── Figure C-1: Nine spurious regressions ────────────────────────────────────
# We search for pairs with R2 > R2_THRESHOLD and DW < DW_THRESHOLD
# to ensure all nine panels look dramatically spurious.
print(f'Searching for 9 spurious pairs (R2>{R2_THRESHOLD}, DW<{DW_THRESHOLD})...')
np.random.seed(None)    # allow different seeds so we explore the distribution

found, attempts = [], 0
while len(found) < 9 and attempts < 200_000:
    attempts += 1
    r = run_regression(T_SPUR, spurious=True)
    if r['r2'] > R2_THRESHOLD and r['dw'] < DW_THRESHOLD:
        found.append(r)
        print(f'  Pair {len(found)}: R2={r["r2"]:.2f}, t={r["t_stat"]:+.1f}, DW={r["dw"]:.2f}')

print(f'Found {len(found)} pairs in {attempts} attempts.')

In [ ]:
# ── Plot the 9 panels ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(14, 12))
fig.suptitle('Figure C-1: Nine Regressions of Independent Random Walks\n'
             'Every relationship shown here is SPURIOUS — no true link exists',
             fontsize=13, fontweight='bold', y=0.98)

for ax, res in zip(axes.flatten(), found):
    # Scatter plot of y vs x
    ax.scatter(res['x'], res['y'], alpha=0.4, s=15,
               color='#9B59B6', edgecolors='#7D3C98', linewidth=0.3)

    # Overlay OLS fitted line
    sort_idx = np.argsort(res['x'])
    ax.plot(res['x'][sort_idx], res['fitted'][sort_idx],
            color='red', linewidth=2.0)

    # Title: significance stars based on |t| — deliberately ironic
    abst = abs(res['t_stat'])
    stars = '***' if abst > 3.29 else ('**' if abst > 2.58 else ('*' if abst > 1.96 else ''))
    sign  = '-' if res['t_stat'] < 0 else '+'
    ax.set_title(f"$R^2$={res['r2']:.2f},  "
                 f"$t$={sign}{abst:.1f}{stars},  "
                 f"DW={res['dw']:.2f}", fontsize=9)
    ax.set_xlabel('$x_t$', fontsize=9)
    ax.set_ylabel('$y_t$', fontsize=9)
    ax.tick_params(labelsize=8)

fig.tight_layout(rect=[0, 0, 1, 0.94])

path = os.path.join(OUT_DIR, 'figC1_spurious_nine_panels.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

---
# Block D — Spurious Regression: Monte Carlo Evidence
### *Textbook reference: Section 1.7.2*

## Background

The nine-panel figure above shows individual realisations. But how common is spurious regression? The Monte Carlo simulation answers this systematically by running thousands of independent regressions and recording the rejection rate, mean $R^2$, and mean DW.

**Key theoretical prediction (Phillips, 1986):** As $T \to \infty$, the rejection rate of a spurious regression at any fixed significance level **does not converge to the nominal level** (e.g., 5%) — it converges to **100%**. The larger the sample, the more certain you are to find a spurious significant result. This is the opposite of what happens with valid regressions.

In [ ]:
# ── Monte Carlo: rejection rates, R2, DW across sample sizes ─────────────────
np.random.seed(42)
SAMPLE_SIZES  = [50, 100, 200, 500]
CRITICAL_VAL  = 1.96    # 5% two-sided critical value under standard normal

mc_summary = {}   # will store {T: {rej_rate, mean_r2, mean_dw}}

for T in SAMPLE_SIZES:
    print(f'  Simulating T={T} ({N_SIMS_SPUR} replications)...')
    reps = [run_regression(T, spurious=True) for _ in range(N_SIMS_SPUR)]

    t_stats = [r['t_stat'] for r in reps]
    r2s     = [r['r2']     for r in reps]
    dws     = [r['dw']     for r in reps]

    # Rejection rate = fraction of t-stats exceeding +/-1.96 in absolute value
    rej_rate = np.mean([abs(t) > CRITICAL_VAL for t in t_stats]) * 100
    mc_summary[T] = {
        'rej_rate': rej_rate,
        'mean_r2':  np.mean(r2s),
        'mean_dw':  np.mean(dws)
    }
    print(f'    → Rej. rate={rej_rate:.1f}%, Mean R2={np.mean(r2s):.3f}, Mean DW={np.mean(dws):.3f}')

print('\nMonte Carlo complete.')

## Figure D-1 — Monte Carlo evidence

Three panels, each telling a different part of the story:

- **(a) Rejection rates**: should be 5% (the dashed line) for a valid test. Spurious regressions with I(1) variables reject dramatically more often — and the rejection rate *increases* with sample size (Phillips's result).
- **(b) Mean $R^2$**: should be near zero for two unrelated variables. Instead it is substantial and stable across sample sizes.
- **(c) Mean DW**: should be near 2 (no autocorrelation). Instead it is near 0, exposing the highly autocorrelated residuals that OLS misinterprets as a good fit.

In [ ]:
# ── Figure D-1: Monte Carlo summary ──────────────────────────────────────────
labels    = [str(T) for T in SAMPLE_SIZES]
rej_rates = [mc_summary[T]['rej_rate'] for T in SAMPLE_SIZES]
mean_r2s  = [mc_summary[T]['mean_r2']  for T in SAMPLE_SIZES]
mean_dws  = [mc_summary[T]['mean_dw']  for T in SAMPLE_SIZES]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel (a): Rejection rates
bars_a = axes[0].bar(labels, rej_rates, color='#E74C6F', edgecolor='white', width=0.6)
axes[0].axhline(5, color='black', linestyle='--', linewidth=1.8,
                label='Nominal 5% level')
axes[0].set_xlabel('Sample Size $T$')
axes[0].set_ylabel('Rejection Rate (%)')
axes[0].set_title('(a) False Rejection Rate\n(should be 5%)')
axes[0].set_ylim(0, 105)
axes[0].legend()
for bar, val in zip(bars_a, rej_rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                 f'{val:.0f}%', ha='center', fontweight='bold', fontsize=11)

# Panel (b): Mean R2
bars_b = axes[1].bar(labels, mean_r2s, color='#6C9BD1', edgecolor='white', width=0.6)
axes[1].set_xlabel('Sample Size $T$')
axes[1].set_ylabel(r'Mean $R^2$')
axes[1].set_title(r'(b) Mean $R^2$'+' from Spurious Regressions\n(should be ≈0)')
axes[1].set_ylim(0, 0.55)
for bar, val in zip(bars_b, mean_r2s):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}', ha='center', fontweight='bold', fontsize=11)

# Panel (c): Mean DW
bars_c = axes[2].bar(labels, mean_dws, color='#2E7D32', edgecolor='white', width=0.6)
axes[2].axhline(2.0, color='red', linestyle='--', linewidth=1.8,
                label='DW = 2 (no autocorrelation)')
axes[2].set_xlabel('Sample Size $T$')
axes[2].set_ylabel('Mean Durbin-Watson')
axes[2].set_title('(c) Mean DW Statistic\n(should be ≈2)')
axes[2].set_ylim(0, 2.6)
axes[2].legend()
for bar, val in zip(bars_c, mean_dws):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.04,
                 f'{val:.2f}', ha='center', fontweight='bold', fontsize=11)

fig.suptitle(f'Figure D-1: Monte Carlo Evidence — Spurious Regression ({N_SIMS_SPUR} replications)\n'
             'As $T$ grows, rejection rates increase (not decrease) — Phillips (1986)',
             fontsize=12, fontweight='bold')
fig.tight_layout()

path = os.path.join(OUT_DIR, 'figD1_mc_evidence.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

## Figure D-2 — Granger-Newbold rule of thumb

For each simulated regression we plot $R^2$ (y-axis) against the Durbin-Watson statistic (x-axis). The diagonal line $R^2 = DW$ is Granger and Newbold's rule of thumb: points **above** the line (high $R^2$, low DW) signal a spurious regression.

Notice that spurious regressions (red dots) cluster in the top-left corner — high $R^2$, low DW — entirely above the diagonal. Valid regressions (blue triangles) scatter below the diagonal with low $R^2$ and DW near 2. The separation is almost perfect, making this rule of thumb a useful quick diagnostic before formal tests.

In [ ]:
# ── Figure D-2: Granger-Newbold scatter ──────────────────────────────────────
np.random.seed(42)

# Spurious: independent random walks
spurious_reps = [run_regression(T_MC, spurious=True)  for _ in range(N_SIMS_SCATTER)]
spur_r2  = [r['r2']  for r in spurious_reps]
spur_dw  = [r['dw']  for r in spurious_reps]

# Valid: stationary, with a true linear relationship
valid_r2, valid_dw = [], []
for _ in range(N_SIMS_SCATTER):
    x = np.random.randn(T_MC)
    y = 0.5 * x + np.random.randn(T_MC)    # true relationship: beta = 0.5
    X = sm.add_constant(x)
    m = sm.OLS(y, X).fit()
    valid_r2.append(m.rsquared)
    valid_dw.append(durbin_watson(m.resid))

fig, ax = plt.subplots(figsize=(10, 7))

# Red shading: the spurious region above the 45° line
dw_fill = np.linspace(0, 2.5, 300)
ax.fill_between(dw_fill, dw_fill, 1.0, alpha=0.12, color='red')
ax.text(0.08, 0.83, 'Spurious Region\n$(R^2 > DW)$',
        transform=ax.transAxes, fontsize=13,
        color='red', fontweight='bold', fontstyle='italic')

# 45° line
dw_line = np.linspace(0, 1.0, 200)
ax.plot(dw_line, dw_line, 'k--', linewidth=2.5, label='$R^2 = DW$ line')

ax.scatter(spur_dw, spur_r2,
           color='#E74C6F', alpha=0.45, s=28,
           edgecolors='#C0392B', linewidth=0.2,
           label='Spurious (two independent I(1) series)')
ax.scatter(valid_dw, valid_r2,
           color='#5DADE2', alpha=0.35, s=28, marker='^',
           edgecolors='#2874A6', linewidth=0.2,
           label='Valid (stationary, true $\\beta=0.5$)')

ax.set_xlabel('Durbin-Watson Statistic', fontsize=12)
ax.set_ylabel('$R^2$', fontsize=12)
ax.set_title("Figure D-2: Granger-Newbold Rule of Thumb\n"
             "If $R^2 > DW$, suspect spurious regression — "
             "Granger & Newbold (1974)",
             fontsize=12, fontweight='bold')
ax.set_xlim(0, 2.5)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=11, loc='lower right')
fig.tight_layout()

path = os.path.join(OUT_DIR, 'figD2_granger_newbold.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

## Figure D-3 — Distribution of t-statistics: spurious vs. valid

This is perhaps the most striking figure in Chapter 1. We compare the distribution of the $t$-statistic on $\hat{\beta}$ from 5,000 regressions in two scenarios:

- **Left panel** (spurious): two independent random walks. The $t$-stats are spread broadly — far outside $\pm 1.96$. The standard $t$-distribution (black dashed) grossly understates the actual uncertainty. At $T=200$, over 75% of spurious regressions would be declared significant at 5%.
- **Right panel** (valid): two independent white noise series. The $t$-stats follow the standard $t$-distribution almost exactly — the test works as intended.

**The mathematical reason (Phillips, 1986):** For I(1) variables, the OLS $t$-statistic does not converge to a standard normal. It converges to a functional of Brownian motion with a distribution that has much heavier tails — explaining why the actual rejection rate under the null is far above 5%.

In [ ]:
# ── Figure D-3: t-statistic distributions ────────────────────────────────────
np.random.seed(42)
print(f'Running {N_SIMS_TSTAT} regressions for each scenario (T={T_MC})...')

spurious_t = [run_regression(T_MC, spurious=True)['t_stat']  for _ in range(N_SIMS_TSTAT)]
valid_t    = [run_regression(T_MC, spurious=False)['t_stat'] for _ in range(N_SIMS_TSTAT)]

# Theoretical t-distribution for comparison
t_grid = np.linspace(-15, 15, 600)
t_pdf  = stats.t.pdf(t_grid, df=T_MC - 2)   # df = T - 2 for simple regression

# Empirical rejection rates at 5%
rej_spur  = np.mean([abs(t) > 1.96 for t in spurious_t]) * 100
rej_valid = np.mean([abs(t) > 1.96 for t in valid_t]) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, t_list, color, title, rej_rate in [
    (axes[0], spurious_t, '#E74C6F',
     f'(a) Spurious Regression\n(two independent random walks, $T={T_MC}$)', rej_spur),
    (axes[1], valid_t,    '#5DADE2',
     f'(b) Valid Regression\n(two independent white noise series, $T={T_MC}$)', rej_valid),
]:
    ax.hist(t_list, bins=60, density=True,
            color=color, alpha=0.6, edgecolor='white',
            label=f'Simulated (rej. rate={rej_rate:.0f}%)')
    ax.plot(t_grid, t_pdf, 'k--', linewidth=2.2,
            label=f'Standard $t$-distribution (df={T_MC-2})')
    for cv, ls in [(-1.96, ':'), (1.96, ':')]:
        ax.axvline(cv, color='blue', linestyle=ls, linewidth=1.5)
    ax.axvline(1.96, color='blue', linestyle=':', linewidth=1.5,
               label='$\\pm 1.96$ critical values')
    ax.set_xlabel('$t$-statistic')
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.set_xlim(-15, 15)
    ax.set_ylim(0, 0.42)
    ax.legend(fontsize=9)

fig.suptitle('Figure D-3: Distribution of OLS $t$-Statistics — Spurious vs. Valid Regression\n'
             'Spurious: heavy tails, standard test breaks down.  Valid: follows theory.',
             fontsize=12, fontweight='bold')
fig.tight_layout()

path = os.path.join(OUT_DIR, 'figD3_tstat_distributions.png')
fig.savefig(path, dpi=150, bbox_inches='tight')
fig.savefig(path.replace('.png','.pdf'), bbox_inches='tight')
plt.show()
print(f'Saved: {path}')

---
## Summary — All figures produced

| Figure | File | Content |
|---|---|---|
| A-1 | `figA1_arma_time_series` | White noise, AR(1), MA(1), ARMA(1,1) over time |
| A-2 | `figA2_arma_acf` | Sample ACF — Box-Jenkins identification |
| A-3 | `figA3_arma_pacf` | Sample PACF — AR order identification |
| B-1 | `figB1_nonstationary_comparison` | Three nonstationary DGPs side by side |
| B-2 | `figB2_nonstationary_acf` | ACF fails for unit roots |
| B-3 | `figB3_levels_vs_differences` | Differencing restores stationarity |
| B-4 | `figB4_variance_growth` | Unbounded variance for I(1), constant for I(0) |
| B-5 | `figB5_fan_chart` | 30 paths of RW vs. deterministic trend |
| C-1 | `figC1_spurious_nine_panels` | Nine individually convincing spurious regressions |
| D-1 | `figD1_mc_evidence` | Rejection rates, $R^2$, DW vs. sample size |
| D-2 | `figD2_granger_newbold` | $R^2$ vs. DW scatter — spurious vs. valid |
| D-3 | `figD3_tstat_distributions` | $t$-stat distribution: spurious vs. valid |

Each figure is saved in both `.png` (for the web) and `.pdf` (for LaTeX/Overleaf) formats.

---

## Exercises

**Exercise 1 — ACF identification challenge**  
Generate an AR(2) process with $\phi_1=0.5$, $\phi_2=0.3$. Plot its ACF and PACF. Where does the PACF cut off? Does it match the theoretical prediction?

**Exercise 2 — Near unit root**  
Change `PHI` to `0.97` and re-run Block A. How similar does the AR(1) ACF look to a random walk's ACF? What does this tell you about the power of unit root tests?

**Exercise 3 — Cointegration preview**  
Modify `run_regression()` so that $y_t = 0.5 x_t + u_t$ where $x_t$ is a random walk and $u_t$ is stationary. Does the spurious regression problem disappear? This is the cointegration case of Section 1.8 — when two I(1) series share a common trend, their regression is valid.

**Exercise 4 — Monte Carlo: power of DW test**  
Using the Monte Carlo infrastructure from Block D, compute the fraction of spurious regressions for which $R^2 > DW$ (Granger-Newbold rule correctly flagged). How does this "power" change with sample size? Compare to the formal DW test at 5%.

---
*Macroeconometrics* | Alessia Paccagnini  
Companion code — Chapter 1 Figure Replication | Last updated: March 2026